In [ ]:
import pandas as pd

In [ ]:
#Upload file "genes.csv" which is derived from HORDE database (https://genome.weizmann.ac.il/horde/app/webroot/index.php/)
from google.colab import files
uploaded = files.upload()

Saving genes.csv to genes.csv


In [ ]:
OR_genes = pd.read_csv("genes.csv", sep = ";")
print(OR_genes.shape)
print(OR_genes.head())

(857, 17)
    Id  Symbol  Family Id Sub Family  Pseduo Gene                 Comments  \
0   51   OR1C1          1          C            0                      NaN   
1  151  OR1X1P          1          X            1  Disrupted by Alu repeat   
2  152  OR1X5P          1          X            1  Disrupted by Alu repeat   
3  166  OR1F12          1          F            0                      NaN   
4  267   OR1J1          1          J            0                      NaN   

   Pseduo Probability                                Nucleotide Sequence  \
0                0.01  ATGGAAAAAAGAAATCTAACAGTTGTCAGGGAATTCGTCCTTCTGG...   
1                0.00  GGGGGTAAAGAGAATGAGACAGGAGTTGGCGAGTTCCTCTTGCTCA...   
2                0.00  ATGTCCAGGGGTAAAGAGAATGAGACAGGAGTTGGCGAGTTCCTCT...   
3                0.04  ATGGAAGGGAAAAATCAAACCAATATCTCTGAATTTCTCCTCCTGG...   
4                0.09  ATGAGCCCTGAGAACCAGAGCAGCGTGTCCGAGTTCCTCCTCCTGG...   

                                 Conceptual Sequence  Clic  Hord

In [ ]:
OR_genes['Pseduo Gene'].value_counts()

,count
Pseduo Gene,
1,466
0,391


In [ ]:
mask_not_pseudo = ~OR_genes['Symbol'].str.strip().str.endswith('P')
print(mask_not_pseudo)

0       True
1      False
2      False
3       True
4       True
       ...  
852     True
853     True
854    False
855     True
856    False
Name: Symbol, Length: 857, dtype: bool


In [ ]:
df_filtered = OR_genes[mask_not_pseudo].copy()

In [ ]:
df_filtered['Pseduo Gene'].value_counts()
print(df_filtered.shape)

(391, 17)


In [ ]:
removed = OR_genes[~mask_not_pseudo]

In [ ]:
removed['Pseduo Gene'].value_counts()

,count
Pseduo Gene,
1,466


In [ ]:
print(df_filtered.columns.tolist())


['Id', 'Symbol', 'Family Id', 'Sub Family', 'Pseduo Gene', 'Comments', 'Pseduo Probability', 'Nucleotide Sequence', 'Conceptual Sequence', 'Clic', 'Horde Id', 'Chromosome', 'Start', 'End', 'Strand', 'Synteny', 'Name']


In [ ]:
print(df_filtered['Pseduo Probability'].dtype)

float64


In [ ]:
PROB_THRESHOLD = 0.31

In [ ]:
mask_low_probability = df_filtered['Pseduo Probability'] < PROB_THRESHOLD

df_filtered_strict = df_filtered[mask_low_probability].copy()
print(f"After probability filter (< {PROB_THRESHOLD}): {len(df_filtered_strict)} lines")

After probability filter (< 0.31): 298 lines


In [ ]:
family_col = 'Family Id'
gene_col = 'Symbol'

gene_sets = (
    df_filtered_strict
    .groupby(family_col)[gene_col]
    .apply(lambda x: sorted(x.dropna().unique().tolist()))
)

print(f"Number of families: {len(gene_sets)}")
for fam, genes in gene_sets.items():
    print(f"  OR_fam{fam}: {len(genes)} genes")

Number of families: 17
  OR_fam1: 22 genes
  OR_fam2: 56 genes
  OR_fam3: 3 genes
  OR_fam4: 35 genes
  OR_fam5: 38 genes
  OR_fam6: 19 genes
  OR_fam7: 11 genes
  OR_fam8: 19 genes
  OR_fam9: 6 genes
  OR_fam10: 29 genes
  OR_fam11: 8 genes
  OR_fam12: 1 genes
  OR_fam13: 9 genes
  OR_fam14: 3 genes
  OR_fam51: 18 genes
  OR_fam52: 19 genes
  OR_fam56: 2 genes


In [ ]:
output_path = "human_OR_families_noPseudo_Probability03.gmt"

with open(output_path, "w", newline="\n") as f:
    for fam_id, genes in gene_sets.items():
        set_name = f"OR_fam{fam_id}"
        description = "na"  # placeholder
        line = "\t".join([set_name, description] + genes)
        f.write(line + "\n")

print(f"GMT file saved: {output_path}")

GMT file saved: human_OR_families_noPseudo_Probability03.gmt


In [ ]:
from google.colab import files

files.download("human_OR_families_noPseudo_Probability03.gmt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>